# LightGCN — Graph-Based Board Game Recommender


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install -q polars pyarrow

In [9]:
import polars as pl
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import ndcg_score
import time
import gc

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


In [ ]:
# --- 1. CONFIGURATION ---
DRIVE_PATH  = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
SPLIT_DIR   = f'{DRIVE_PATH}/processed_data/final_foundation/splits'

# Sample fraction to stay within GPU memory (≈15% ≈ 2.25M train rows)
SAMPLE_FRAC = 0.15

# Model hyper-parameters
EMB_DIM     = 64
N_LAYERS    = 3
LR          = 5e-3
WEIGHT_DECAY= 1e-5
BATCH_SIZE  = 4096
N_EPOCHS    = 30
PATIENCE    = 5       # early stopping on val RMSE

In [ ]:
# --- 2. LOAD & SAMPLE DATA ---
print('Loading data...')

train_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/train.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
      .sample(fraction=SAMPLE_FRAC, seed=42)
)

val_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/val.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
)

test_df = (
    pl.scan_parquet(f'{SPLIT_DIR}/test.parquet')
      .select(['user_id', 'item_id', 'Rating'])
      .collect()
)

print(f'  Train rows : {len(train_df):,}')
print(f'  Val rows   : {len(val_df):,}')
print(f'  Test rows  : {len(test_df):,}')

Loading data...
  Train rows : 2,239,383
  Val rows   : 1,856,255
  Test rows  : 2,124,050


In [ ]:
# --- 3. CONTIGUOUS ID MAPPING ---
# LightGCN needs 0-indexed, contiguous integer IDs.

unique_users = train_df['user_id'].unique().sort().to_list()
unique_items = train_df['item_id'].unique().sort().to_list()

user2idx = {u: i for i, u in enumerate(unique_users)}
item2idx = {it: i for i, it in enumerate(unique_items)}

N_USERS = len(user2idx)
N_ITEMS = len(item2idx)
print(f'  Unique users in train : {N_USERS:,}')
print(f'  Unique items in train : {N_ITEMS:,}')

GLOBAL_MEAN = train_df['Rating'].mean()
print(f'  Global mean rating    : {GLOBAL_MEAN:.4f}')

def map_ids(df, u2i=user2idx, it2i=item2idx):
    """Map original IDs → contiguous indices; drop rows with unseen IDs."""
    df = df.with_columns([
        pl.col('user_id').replace(u2i, default=None).alias('uid'),
        pl.col('item_id').replace(it2i, default=None).alias('iid'),
    ]).drop_nulls(subset=['uid', 'iid'])
    return df

train_mapped = map_ids(train_df)
val_mapped   = map_ids(val_df)
test_mapped  = map_ids(test_df)

print(f'  Val rows after ID filter  : {len(val_mapped):,}')
print(f'  Test rows after ID filter : {len(test_mapped):,}')

  Unique users in train : 253,190
  Unique items in train : 21,880
  Global mean rating    : 7.1173


/tmp/ipykernel_4044/183677431.py:22: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col('user_id').replace(u2i, default=None).alias('uid'),
/tmp/ipykernel_4044/183677431.py:23: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col('item_id').replace(it2i, default=None).alias('iid'),


  Val rows after ID filter  : 1,823,522
  Test rows after ID filter : 1,953,127


In [ ]:
# --- 4. NORMALISED LAPLACIAN (Sparse, CPU) ---
# LightGCN uses the symmetrically normalised adjacency of the bipartite graph:

def build_norm_adj(train_df_mapped, n_users, n_items):
    uids = train_df_mapped['uid'].to_numpy()
    iids = train_df_mapped['iid'].to_numpy()
    n    = n_users + n_items

    # Upper-right block: users -> items
    rows = np.concatenate([uids, iids + n_users])
    cols = np.concatenate([iids + n_users, uids])
    data = np.ones(len(rows), dtype=np.float32)

    A = sp.csr_matrix((data, (rows, cols)), shape=(n, n))

    # D^{-1/2}
    deg    = np.array(A.sum(axis=1)).flatten()
    d_inv  = np.where(deg > 0, deg ** -0.5, 0.0)
    D_inv  = sp.diags(d_inv)
    A_hat  = D_inv @ A @ D_inv

    # Convert to torch sparse tensor
    A_coo = A_hat.tocoo()
    indices = torch.from_numpy(
        np.vstack([A_coo.row, A_coo.col]).astype(np.int64)
    )
    values  = torch.from_numpy(A_coo.data.astype(np.float32))
    sparse_A = torch.sparse_coo_tensor(indices, values, (n, n))
    return sparse_A

print('Building normalised Laplacian...')
t0 = time.time()
norm_adj = build_norm_adj(train_mapped, N_USERS, N_ITEMS)
norm_adj = norm_adj.to(DEVICE)
print(f'  Done in {time.time()-t0:.1f}s  |  nnz={norm_adj._nnz():,}')

Building normalised Laplacian...
  Done in 1.0s  |  nnz=4,478,766


In [14]:
# --- 5. LIGHTGCN MODEL ---

class LightGCN(nn.Module):
    """
    LightGCN (He et al., 2020) adapted for explicit rating prediction.

    Forward pass:
      E^(0) = concat(user_emb, item_emb)          # base embeddings
      E^(k) = A_hat * E^(k-1)                     # graph convolution (no weights)
      E_final = mean(E^(0), ..., E^(K))            # layer aggregation

    Rating prediction:
      r_hat = mu + b_u + b_i + dot(e_u, e_i)
    """

    def __init__(self, n_users, n_items, emb_dim, n_layers, global_mean):
        super().__init__()
        self.n_users     = n_users
        self.n_items     = n_items
        self.n_layers    = n_layers
        self.global_mean = global_mean

        # Base embeddings
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        # Bias terms
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def propagate(self, norm_adj):
        """Run LightGCN propagation and return mean-pooled user & item embeddings."""
        E0 = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)  # (N+M, d)
        all_embs = [E0]
        E = E0
        for _ in range(self.n_layers):
            E = torch.sparse.mm(norm_adj, E)
            all_embs.append(E)
        E_final = torch.stack(all_embs, dim=1).mean(dim=1)  # layer-mean
        user_final = E_final[:self.n_users]
        item_final = E_final[self.n_users:]
        return user_final, item_final

    def forward(self, uid, iid, norm_adj):
        user_final, item_final = self.propagate(norm_adj)
        e_u = user_final[uid]                       # (B, d)
        e_i = item_final[iid]                       # (B, d)
        dot  = (e_u * e_i).sum(dim=1)               # (B,)
        b_u  = self.user_bias(uid).squeeze(1)       # (B,)
        b_i  = self.item_bias(iid).squeeze(1)       # (B,)
        r_hat = self.global_mean + b_u + b_i + dot  # (B,)
        return r_hat

In [15]:
# --- 6. TORCH DATASET ---

class RatingDataset(Dataset):
    def __init__(self, df_mapped):
        self.uids    = torch.tensor(df_mapped['uid'].to_numpy(),    dtype=torch.long)
        self.iids    = torch.tensor(df_mapped['iid'].to_numpy(),    dtype=torch.long)
        self.ratings = torch.tensor(df_mapped['Rating'].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.uids[idx], self.iids[idx], self.ratings[idx]

train_loader = DataLoader(
    RatingDataset(train_mapped), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    RatingDataset(val_mapped), batch_size=BATCH_SIZE * 4, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    RatingDataset(test_mapped), batch_size=BATCH_SIZE * 4, shuffle=False,
    num_workers=2, pin_memory=True
)
print(f'Batches per epoch: {len(train_loader):,}')

Batches per epoch: 547


In [16]:
# --- 7. TRAINING ---

model = LightGCN(
    n_users=N_USERS,
    n_items=N_ITEMS,
    emb_dim=EMB_DIM,
    n_layers=N_LAYERS,
    global_mean=GLOBAL_MEAN
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

def evaluate(loader):
    model.eval()
    se, n = 0.0, 0
    with torch.no_grad():
        for uid, iid, rating in loader:
            uid, iid, rating = uid.to(DEVICE), iid.to(DEVICE), rating.to(DEVICE)
            preds = model(uid, iid, norm_adj).clamp(1, 10)
            se += ((rating - preds) ** 2).sum().item()
            n  += len(rating)
    return (se / n) ** 0.5

best_val_rmse = float('inf')
patience_ctr  = 0
best_state    = None

print(f'Training LightGCN for up to {N_EPOCHS} epochs...\n')
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    t0 = time.time()
    total_loss, n_batches = 0.0, 0

    for uid, iid, rating in train_loader:
        uid, iid, rating = uid.to(DEVICE), iid.to(DEVICE), rating.to(DEVICE)
        optimizer.zero_grad()
        preds = model(uid, iid, norm_adj)
        loss  = criterion(preds, rating)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1

    scheduler.step()
    val_rmse = evaluate(val_loader)
    avg_loss = total_loss / n_batches
    elapsed  = time.time() - t0

    print(f'Epoch {epoch:02d} | Loss={avg_loss:.4f} | Val RMSE={val_rmse:.4f} | {elapsed:.0f}s')

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
            break

print(f'\nBest Validation RMSE: {best_val_rmse:.4f}')

Training LightGCN for up to 30 epochs...

Epoch 01 | Loss=1.9147 | Val RMSE=1.3064 | 46s
Epoch 02 | Loss=1.6185 | Val RMSE=1.2587 | 44s
Epoch 03 | Loss=1.5170 | Val RMSE=1.2389 | 44s
Epoch 04 | Loss=1.4691 | Val RMSE=1.2295 | 43s
Epoch 05 | Loss=1.4438 | Val RMSE=1.2243 | 43s
Epoch 06 | Loss=1.4295 | Val RMSE=1.2214 | 43s
Epoch 07 | Loss=1.4209 | Val RMSE=1.2197 | 43s
Epoch 08 | Loss=1.4157 | Val RMSE=1.2185 | 43s
Epoch 09 | Loss=1.4123 | Val RMSE=1.2179 | 43s
Epoch 10 | Loss=1.4103 | Val RMSE=1.2174 | 44s
Epoch 11 | Loss=1.3931 | Val RMSE=1.2170 | 44s
Epoch 12 | Loss=1.3926 | Val RMSE=1.2169 | 45s
Epoch 13 | Loss=1.3924 | Val RMSE=1.2168 | 45s
Epoch 14 | Loss=1.3922 | Val RMSE=1.2167 | 44s
Epoch 15 | Loss=1.3919 | Val RMSE=1.2166 | 43s
Epoch 16 | Loss=1.3918 | Val RMSE=1.2166 | 43s
Epoch 17 | Loss=1.3916 | Val RMSE=1.2166 | 43s
Epoch 18 | Loss=1.3914 | Val RMSE=1.2165 | 43s
Epoch 19 | Loss=1.3914 | Val RMSE=1.2165 | 43s
Epoch 20 | Loss=1.3913 | Val RMSE=1.2165 | 43s
Epoch 21 | Loss=1.

In [17]:
# --- 8. TEST EVALUATION ---
model.load_state_dict(best_state)

test_rmse = evaluate(test_loader)
print('-' * 45)
print(f'  LIGHTGCN FINAL TEST RMSE: {test_rmse:.4f}')
print('-' * 45)

---------------------------------------------
  LIGHTGCN FINAL TEST RMSE: 1.2229
---------------------------------------------


In [20]:
import random

print('\nCalculating NDCG@10 & Catalog Coverage (500 sampled users)...')
t0 = time.time()

model.eval()
with torch.no_grad():
    user_embs, item_embs = model.propagate(norm_adj)
    user_embs   = user_embs.cpu()
    item_embs   = item_embs.cpu()
    user_biases = model.user_bias.weight.squeeze(1).cpu()   # (N_USERS,)
    item_biases = model.item_bias.weight.squeeze(1).cpu()   # (N_ITEMS,)

# Build test lookup: uid -> [(iid, rating), ...]
test_by_user = {}
for row in test_mapped.iter_rows(named=True):
    test_by_user.setdefault(row['uid'], []).append((row['iid'], row['Rating']))

sample_uids = random.sample(list(test_by_user.keys()), min(500, len(test_by_user)))

# NDCG@10
ndcg_scores = []
for uid in sample_uids:
    pairs = test_by_user[uid]
    if len(pairs) < 2:
        continue
    iids_u    = torch.tensor([p[0] for p in pairs])
    ratings_u = [p[1] for p in pairs]
    e_u   = user_embs[uid].unsqueeze(0)
    e_i   = item_embs[iids_u]
    preds = (
        GLOBAL_MEAN
        + user_biases[uid].item()
        + item_biases[iids_u]          # already 1D
        + (e_u * e_i).sum(dim=1)
    ).numpy()
    ndcg_scores.append(ndcg_score([ratings_u], [preds], k=10))

# Catalog Coverage
recommended = set()
for uid in sample_uids:
    e_u   = user_embs[uid].unsqueeze(0)
    preds = (
        GLOBAL_MEAN
        + user_biases[uid].item()
        + item_biases                  # (N_ITEMS,)
        + (e_u * item_embs).sum(dim=1)
    ).numpy()
    top10 = np.argsort(-preds)[:10]
    recommended.update(top10.tolist())

coverage = len(recommended) / N_ITEMS
elapsed  = time.time() - t0

print('-' * 45)
print(f'  LIGHTGCN NDCG@10 : {np.mean(ndcg_scores):.4f}')
print(f'  LIGHTGCN COVERAGE: {coverage:.2%}')
print(f'  (Calculated in {elapsed:.1f}s)')
print('-' * 45)


Calculating NDCG@10 & Catalog Coverage (500 sampled users)...
---------------------------------------------
  LIGHTGCN NDCG@10 : 0.9677
  LIGHTGCN COVERAGE: 0.05%
  (Calculated in 5.4s)
---------------------------------------------


In [21]:
# --- 10. SAVE MODEL ---
import os

SAVE_DIR   = f'{DRIVE_PATH}/processed_data'
MODEL_PATH = f'{SAVE_DIR}/lightgcn_model.pt'

torch.save({
    'model_state': best_state,
    'user2idx'   : user2idx,
    'item2idx'   : item2idx,
    'config': {
        'emb_dim'    : EMB_DIM,
        'n_layers'   : N_LAYERS,
        'n_users'    : N_USERS,
        'n_items'    : N_ITEMS,
        'global_mean': GLOBAL_MEAN,
    }
}, MODEL_PATH)

print(f'Model saved to: {MODEL_PATH}')
print()
print('=== FINAL SUMMARY ===')
print(f'  Train rows used : {len(train_mapped):,}  (sample_frac={SAMPLE_FRAC})')
print(f'  Users / Items   : {N_USERS:,} / {N_ITEMS:,}')
print(f'  Best Val RMSE   : {best_val_rmse:.4f}')
print(f'  Test RMSE       : {test_rmse:.4f}')

Model saved to: /content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/processed_data/lightgcn_model.pt

=== FINAL SUMMARY ===
  Train rows used : 2,239,383  (sample_frac=0.15)
  Users / Items   : 253,190 / 21,880
  Best Val RMSE   : 1.2164
  Test RMSE       : 1.2229


In [23]:
# --- 11. BASELINE COMPARISON TABLE ---
baselines = [
    ('Popularity-Based',   'Baseline',  None),
    ('SVD',               'Baseline',  None),
    ('LightGCN',          "Darshan's", round(test_rmse, 4)),
]

print('\n{:<22} {:<12} {}'.format('Model', 'Author', 'Test RMSE'))
print('-' * 45)
for name, author, rmse in baselines:
    rmse_str = f'{rmse:.4f}' if rmse else 'In the report'
    print(f'{name:<22} {author:<12} {rmse_str}')


Model                  Author       Test RMSE
---------------------------------------------
Popularity-Based       Baseline     In the report
SVD                    Baseline     In the report
LightGCN               Darshan's    1.2229
